In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

import os
import sys
import importlib

project_pth = os.path.join(os.getcwd(), '..', '..')
sys.path.append(project_pth)

# Force reload to pick up latest changes from the .py file
import utils.transformations
importlib.reload(utils.transformations)

from utils.transformations import reusable

## **DimUser**

### AUTOLOADER

**Defined Schema manually**

In [0]:
# Step 1: Define exact schema
schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("user_name", StringType(), True),
    StructField("country", StringType(), True),
    StructField("subscription_type", StringType(), True),
    StructField("start_date", DateType(), True),
    StructField("end_date", DateType(), True),
    StructField("updated_at", TimestampType(), True),
    StructField("_rescued_data", StringType(), True)
])

**Reading the data from the bronze layer**

In [0]:
df_user = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .schema(schema) \
    .load("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/DimUser")


**Data Validation check**

In [0]:
display(spark.read.parquet("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/DimUser").limit(10))

**Transforming the data - using Upper()**

In [0]:
df_user = df_user.withColumn("user_name", upper(col("user_name")))

In [0]:
display(spark.read.schema(schema).parquet("abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimUser/data"))

**Dropping specific column(s) and Duplicates**

In [0]:
df_user_obj = reusable()

df_user = df_user_obj.dropColumns(df_user, ['_rescued_data'])
df_user = df_user.dropDuplicates(['user_id'])

display(spark.read.parquet("abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimUser/data"))

**Writing the Transformed Data into the silver layer**

In [0]:
# Step 1: Clear old checkpoint and data
dbutils.fs.rm("abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimUser/checkpoint", recurse=True)
dbutils.fs.rm("abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimUser/data", recurse=True)

# Step 2: Rerun with delta format
query = df_user.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimUser/checkpoint") \
    .trigger(once=True) \
    .toTable("`spotify-catalog`.silver.DimUser")

query.awaitTermination()

## **DimArtist**

In [0]:
df_art = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet") \
    .schema(schema_art) \
    .load("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/DimArtist")

display(spark.read.parquet("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/DimArtist"))


In [0]:
# Define DimArtist schema
schema_art = StructType([
    StructField("artist_id", IntegerType(), True),
    StructField("artist_name", StringType(), True),
    StructField("genre", StringType(), True),
    StructField("country", StringType(), True),
    StructField("updated_at", TimestampType(), True)
])

In [0]:
#Transform
df_art = df_art.withColumn("artist_name", upper(col("artist_name")))

# Step 3: Write stream to silver
query = df_art.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimArtist/checkpoint") \
    .trigger(once=True) \
    .toTable("`spotify-catalog`.silver.DimArtist")

query.awaitTermination()

# Step 4: Preview
display(spark.read.format("delta").load("abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimArtist/data"))


## **DimTrack**

In [0]:
df_trc = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet")\
    .schema(schema_track)\
    .load("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/DimTrack")

display(spark.read.parquet("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/DimTrack"))

In [0]:
schema_track = StructType([
    StructField("track_id", IntegerType(), True),
    StructField("track_name", StringType(), True),
    StructField("artist_id", IntegerType(), True),
    StructField("album_name", StringType(), True),
    StructField("duration_sec", IntegerType(), True),
    StructField("release_date", DateType(), True),
    StructField("updated_at", TimestampType(), True)
])

In [0]:
#Transform
df_trc = df_trc.withColumn("track_name", upper(col("track_name")))

# Step 3: Write stream to silver
query = df_trc.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimTrack/checkpoint") \
    .trigger(once=True) \
    .toTable("`spotify-catalog`.silver.DimTrack")

query.awaitTermination()

In [0]:
df_trc = df_trc.withColumn("duration_Flag", when(col("duration_sec") < 150, "low") \
                                           .when(col("duration_sec") < 300, "medium")\
                                           .otherwise("high"))

df_trc = df_trc.withColumn("track_name", regexp_replace(col("track_name"), "-", " "))

display(spark.read.table("`spotify-catalog`.silver.DimTrack"))

In [0]:
dbutils.fs.rm("abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimTrack/checkpoint", recurse=True)
spark.sql("DROP TABLE IF EXISTS `spotify-catalog`.silver.DimTrack")

## **DimDate**

In [0]:
df_date = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet")\
    .schema(schema_date)\
    .load("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/DimDate")

display(spark.read.parquet("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/DimDate"))

In [0]:
schema_date = StructType([
    StructField("date_key", IntegerType(), True),
    StructField("date", DateType(), True),
    StructField("day", IntegerType(), True),
    StructField("month", IntegerType(), True),
    StructField("year", IntegerType(), True),
    StructField("weekday", StringType(), True)
])

In [0]:
query = df_date.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@azureprojectstorage01.dfs.core.windows.net/DimDate/checkpoint") \
    .trigger(availableNow=True) \
    .toTable("`spotify-catalog`.silver.DimDate")

query.awaitTermination()


display(spark.read.table("`spotify-catalog`.silver.DimDate"))

## **FactStream**

In [0]:
schema_fact = StructType([
    StructField("stream_id", LongType(), True),
    StructField("user_id", LongType(), True),
    StructField("track_id", LongType(), True),
    StructField("date_key", LongType(), True),
    StructField("listen_duration", LongType(), True),
    StructField("device_type", StringType(), True),
    StructField("stream_timestamp", TimestampType(), True)
])

In [0]:
df_fact = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "parquet")\
    .schema(schema_fact)\
    .load("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/FactStream")

display(spark.read.parquet("abfss://bronze@azureprojectstorage01.dfs.core.windows.net/FactStream"))

In [0]:
query = df_fact.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "abfss://silver@azureprojectstorage01.dfs.core.windows.net/FactStream/checkpoint") \
    .trigger(availableNow=True) \
    .toTable("`spotify-catalog`.silver.FactStream")

query.awaitTermination()

display(spark.read.table("`spotify-catalog`.silver.FactStream"))